In [1]:
# point cloud testing
"""
Create points in a space
Each point is a system that contains a certain number of planets
    The more planets the more total land for building stuff
    The quality of the system is a rough estimate of the amount of resources in the system
This needs to be made so that a target can be selected and a new system can come under control

To move to a new system a transportation system needs to be set up
    Transportation determines the speed at which resources can be moved from system to system
    Researching and developing transportation methods 

Planets
    Planets are created from distributions determining the following stats
    Land
    Water
    Ore
    Oil
"""

'\nCreate points in a space\nEach point is a system that contains a certain number of planets\n    The more planets the more total land for building stuff\n    The quality of the system is a rough estimate of the amount of resources in the system\nThis needs to be made so that a target can be selected and a new system can come under control\n\nTo move to a new system a transportation system needs to be set up\n    Transportation determines the speed at which resources can be moved from system to system\n    Researching and developing transportation methods \n\nPlanets\n    Planets are created from distributions determining the following stats\n    Land\n    Water\n    Ore\n    Oil\n'

In [ ]:
import numpy as np
from dataclasses import dataclass, field
from typing import Literal, Optional
from enum import IntEnum

# ------------------------------------------------------------------
# DISTRIBUTION HELPER
# ------------------------------------------------------------------

DISTRIBUTION = Literal["normal", "uniform", "lognormal", "exponential", "beta"]

def sample_distribution(
    dist:   DISTRIBUTION,
    low:    float = 0.0,
    high:   float = 1.0,
    mean:   float = 0.5,
    std:    float = 0.1,
    *,
    scale:  float = 1.0,
    size:   int   = 1,
    clip:   Optional[tuple[float, float]] = None,
    rng:    Optional[np.random.Generator] = None
) -> np.ndarray:
    """
    Draw `size` samples from a named distribution, then scale and optionally clip.

    Parameters
    ----------
    dist    : which distribution to use
    low/high: bounds for uniform; also used as clip range if clip=None for uniform
    mean/std: for normal and lognormal (lognormal treats these as the underlying normal params)
    scale   : multiply all samples by this — easy way to convert a [0,1] draw into real units
    size    : number of samples
    clip    : (min, max) to clamp output — None means no clamping
    rng     : optional numpy Generator for reproducibility

    Returns
    -------
    np.ndarray of shape (size,)

    Distribution quick-reference
    ----------------------------
    uniform     -> flat chance across [low, high]. Good for resources with no preferred value.
    normal      -> bell curve around mean with spread std. Good for "typical" quantities.
    lognormal   -> right-skewed, always positive. Good for wealth/resource hoards where
                   most planets have little but a few have enormous reserves.
    exponential -> heavy right tail, peaks at 0. Good for rare windfalls.
    beta        -> flexible [0,1] shape via mean/std. Good for quality/efficiency scores.
                   Internally converts mean+std -> alpha/beta params.
    """
    rng = rng or np.random.default_rng()

    match dist:
        case "uniform":
            samples = rng.uniform(low, high, size=size)

        case "normal":
            samples = rng.normal(mean, std, size=size)

        case "lognormal":
            # mean/std refer to the underlying normal, not the output distribution
            samples = rng.lognormal(mean, std, size=size)

        case "exponential":
            # mean controls the scale (1/lambda)
            samples = rng.exponential(mean, size=size)

        case "beta":
            # Convert mean+std to alpha/beta params
            var = std ** 2
            var = min(var, mean * (1 - mean) - 1e-6)  # keep params valid
            alpha = mean * (mean * (1 - mean) / var - 1)
            beta  = (1 - mean) * (mean * (1 - mean) / var - 1)
            alpha = max(alpha, 0.01)
            beta  = max(beta,  0.01)
            samples = rng.beta(alpha, beta, size=size)

        case _:
            raise ValueError(f"Unknown distribution: {dist!r}")

    samples = samples * scale
    if clip is not None:
        samples = np.clip(samples, clip[0], clip[1])

    return samples


# ------------------------------------------------------------------
# RESOURCE TYPES
# ------------------------------------------------------------------

class ResourceType(IntEnum):
    MINERALS      = 0
    ENERGY        = 1
    ORGANICS      = 2
    RARE_MATS     = 3

@dataclass
class ResourceConfig:
    """
    Defines how one resource type is generated.
    All sample_distribution kwargs can be set here.
    """
    dist:  DISTRIBUTION  = "lognormal"
    mean:  float         = 1.0
    std:   float         = 0.5
    scale: float         = 100.0
    clip:  Optional[tuple[float, float]] = (0.0, None)

    # How much planet size and quality amplify the base draw
    size_exponent:    float = 1.2   # output *= size ** size_exponent
    quality_exponent: float = 1.5   # output *= quality ** quality_exponent


# Default configs — easy to override per planet/system
DEFAULT_RESOURCE_CONFIGS: dict[ResourceType, ResourceConfig] = {
    ResourceType.MINERALS:  ResourceConfig(dist="lognormal",   mean=1.0, std=0.6,  scale=200.0,  size_exponent=1.3, quality_exponent=1.2), # things like stone and ore
    ResourceType.ENERGY:    ResourceConfig(dist="exponential", mean=1.0, std=0.5,  scale=150.0,  size_exponent=1.0, quality_exponent=1.8), # non renewable energy like coal
    ResourceType.ORGANICS:  ResourceConfig(dist="beta",        mean=0.3, std=0.15, scale=180.0,  size_exponent=1.1, quality_exponent=2.0), # animals, plants, lumber, etc.
    ResourceType.RARE_MATS: ResourceConfig(dist="exponential", mean=0.4, std=0.3,  scale=50.0,   size_exponent=0.8, quality_exponent=2.5)  # uranium, platinum, tungsten, cobalt, gold, silver, titanium, etc.
}


# ------------------------------------------------------------------
# PLANET
# ------------------------------------------------------------------

@dataclass
class Planet:
    """
    size    : float in [0.1, 10.0] — relative planet size
    quality : float in [0, 5]      — overall resource richness score
    rng     : shared Generator for reproducibility
    """
    size:    float
    quality: float
    rng:     np.random.Generator = field(default_factory=np.random.default_rng, repr=False)
    resource_configs: dict[ResourceType, ResourceConfig] = field(
        default_factory=lambda: DEFAULT_RESOURCE_CONFIGS, repr=False
    )

    resources: dict[ResourceType, float] = field(init=False)

    def __post_init__(self):
        self.resources = self._generate_resources()

    def _generate_resources(self) -> dict[ResourceType, float]:
        out = {}
        q_norm = self.quality / 5.0   # normalise quality to [0, 1]

        for rtype, cfg in self.resource_configs.items():
            # Base draw from chosen distribution
            base = sample_distribution(
                dist  = cfg.dist,
                mean  = cfg.mean,
                std   = cfg.std,
                scale = cfg.scale,
                size  = 1,
                clip  = cfg.clip,
                rng   = self.rng
            )[0]

            # Scale by planet size and quality
            value = base * (self.size ** cfg.size_exponent) * (q_norm ** cfg.quality_exponent)

            # Quality=0 planets produce nothing
            out[rtype] = round(value if q_norm > 0 else 0.0, 2)

        return out

    def summary(self) -> str:
        lines = [f"  Planet  size={self.size:.2f}  quality={self.quality:.1f}/5"]
        for rtype, val in self.resources.items():
            lines.append(f"    {rtype.name:<12} {val:>10.2f}")
        return "\n".join(lines)


# ------------------------------------------------------------------
# SOLAR SYSTEM
# ------------------------------------------------------------------

@dataclass
class SolarSystem:
    """
    position      : (x, y, z) in arbitrary space units
    n_planets     : number of planets to generate
    seed          : optional int for reproducibility
    quality_dist  : how planet quality scores are drawn
    size_dist     : how planet sizes are drawn
    """
    position:     tuple[float, float, float]
    n_planets:    int   = 5
    seed:         Optional[int] = None
    quality_dist: dict  = field(default_factory=lambda: dict(dist="beta",    mean=0.4, std=0.2, scale=5.0, clip=(0.0, 5.0)))
    size_dist:    dict  = field(default_factory=lambda: dict(dist="uniform", low=0.1,  high=10.0))

    planets:      list[Planet] = field(init=False)
    rng:          np.random.Generator = field(init=False, repr=False)

    def __post_init__(self):
        self.rng     = np.random.default_rng(self.seed)
        self.planets = self._generate_planets()

    def _generate_planets(self) -> list[Planet]:
        qualities = sample_distribution(**self.quality_dist, size=self.n_planets, rng=self.rng)
        sizes     = sample_distribution(**self.size_dist,    size=self.n_planets, rng=self.rng)

        return [
            Planet(size=float(s), quality=float(q), rng=self.rng)
            for s, q in zip(sizes, qualities)
        ]

    def total_resources(self) -> dict[ResourceType, float]:
        totals = {r: 0.0 for r in ResourceType}
        for planet in self.planets:
            for rtype, val in planet.resources.items():
                totals[rtype] += val
        return totals

    def summary(self) -> str:
        lines = [f"SolarSystem @ {self.position}  ({len(self.planets)} planets)"]
        for i, p in enumerate(self.planets):
            lines.append(f"  -- Planet {i+1} --")
            lines.append(p.summary())
        lines.append("  -- System Totals --")
        for rtype, val in self.total_resources().items():
            lines.append(f"    {rtype.name:<12} {val:>10.2f}")
        return "\n".join(lines)
    

In [3]:
# Use the shit I made (claude made) above to test it out
pos_1 = (1.0, 4.2, -3.1)
planet_num = 6
seed = 42
sys1 = SolarSystem(position=pos_1, n_planets=planet_num, seed=seed)
print(sys1.summary())

# Richer system — high quality beta dist skewed toward 4-5
rich_system = SolarSystem(
    position=(10.0, 0.0, 5.0),
    n_planets=4,
    quality_dist=dict(dist="beta", mean=0.75, std=0.1, scale=5.0, clip=(0.0, 5.0)),
    size_dist=dict(dist="lognormal", mean=1.5, std=0.4, scale=1.0, clip=(0.5, 12.0)),
    seed=67
    )
print(rich_system.summary())

# Override resource configs for a mineral-rich system
from copy import deepcopy
mineral_configs = deepcopy(DEFAULT_RESOURCE_CONFIGS)
mineral_configs[ResourceType.MINERALS].scale = 800.0
mineral_configs[ResourceType.MINERALS].quality_exponent = 3.0

sys2 = SolarSystem(position=(2.0, 2.0, 2.0), n_planets=3, seed=1)
for planet in sys2.planets:
    planet.resource_configs = mineral_configs

print(sys2.summary())

SolarSystem @ (1.0, 4.2, -3.1)  (6 planets)
  -- Planet 1 --
  Planet  size=4.72  quality=1.7/5
    MINERALS         819.16
    ENERGY            23.96
    ORGANICS          43.15
    RARE_MATS          4.05
  -- Planet 2 --
  Planet  size=0.53  quality=2.1/5
    MINERALS         121.94
    ENERGY            21.49
    ORGANICS           5.11
    RARE_MATS          0.37
  -- Planet 3 --
  Planet  size=1.63  quality=2.6/5
    MINERALS         798.82
    ENERGY            12.72
    ORGANICS          29.69
    RARE_MATS         23.66
  -- Planet 4 --
  Planet  size=6.86  quality=2.1/5
    MINERALS        1906.87
    ENERGY            59.88
    ORGANICS          31.24
    RARE_MATS         14.20
  -- Planet 5 --
  Planet  size=7.47  quality=2.8/5
    MINERALS        1362.01
    ENERGY          2125.23
    ORGANICS         129.65
    RARE_MATS          1.06
  -- Planet 6 --
  Planet  size=9.68  quality=3.3/5
    MINERALS        4714.34
    ENERGY           417.64
    ORGANICS         342.61


In [ ]:
"""
Create the faction stuff
- First goal will be to create a faction that only has access to one system
- Each planet will be its own colony and the goal of the faction is to balance the outputs from the colonies until there is
  a net positive throughout the whole system
- Resources to be tracking
    - Minerals
    - Energy
    - Organics
    - Rare Material
    - Building Material
    - Science
        - Military
        - Agriculture
        - Gathering
        - Transport
        - Energy
    - Military
        - Defense
        - Offense (will add this later once multiple factions are present)
Buildings will harvest materials from the planet or create new materials
    Mine - Harvests minerals from the planet
    PowerPlant - creates energy by using a different resource
    Farm - creates organics which support population
    Lab - create research which provides building upgrades
    Shipyard - creates ships that transport material betweween colonies
    Defense - Creates Defense for the planet
"""